In [ ]:
import os
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import torch.optim as optim
from sklearn.model_selection import train_test_split

# --- Configuration ---
TRAIN_DIR = "/kaggle/input/datasets/macharning/chula-parasite-dataset/Chula-ParasiteEgg-11/Chula-ParasiteEgg-11/Chula-ParasiteEgg-11/data"
LABEL_PATH = "/kaggle/input/datasets/macharning/chula-parasite-dataset/Chula-ParasiteEgg-11/Chula-ParasiteEgg-11/Chula-ParasiteEgg-11/labels.json"
TEST_DIR = "/kaggle/input/competitions/super-ai-engineer-season-6-parasite-eggs/test_set/test"
SUB_PATH = "/kaggle/input/competitions/super-ai-engineer-season-6-parasite-eggs/sample_submission.csv"
MODEL_NAME = 'dinov2_vitl14'
NUM_CLASSES = 12
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 5
LR = 5e-5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Custom Dataset ---
class ParasiteDataset(Dataset):
    def __init__(self, data_list, root_dir, transform=None):
        self.data_list = data_list
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        filename = self.data_list[idx]['filename']
        img_path = os.path.join(self.root_dir, filename)
        image = Image.open(img_path).convert('RGB')
        label = int(self.data_list[idx]['label'])

        if self.transform:
            image = self.transform(image)
        return image, label

# --- Preprocessing & Augmentation ---
transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(180),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- Model Definition ---
class DinoV2Classifier(nn.Module):
    def __init__(self, num_classes):
        super(DinoV2Classifier, self).__init__()
        self.backbone = torch.hub.load('facebookresearch/dinov2', MODEL_NAME)
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

# --- Data Preparation ---
with open(LABEL_PATH, 'r') as f:
    raw_labels = json.load(f)

train_data = []

# Scenario 1: The JSON is in COCO format
if 'images' in raw_labels and 'annotations' in raw_labels:
    print("Detected COCO JSON format. Parsing...")

    # 1. Map Image IDs to their actual filenames
    img_dict = {img['id']: img['file_name'] for img in raw_labels['images']}

    # 2. Map Category IDs (labels) to the filenames
    seen_images = set()
    for ann in raw_labels['annotations']:
        img_id = ann['image_id']

        # We only take the first label per image for basic classification
        if img_id in img_dict and img_id not in seen_images:
            train_data.append({
                'filename': img_dict[img_id],
                'label': int(ann['category_id'])
            })
            seen_images.add(img_id)

# Scenario 2: It's a flat dictionary, but labels are wrapped in lists e.g., {"img.jpg": [1]}
else:
    print("Detected flat dictionary format. Parsing...")
    for k, v in raw_labels.items():
        # If the label is inside a list, extract the first element
        label_val = v[0] if isinstance(v, list) else v
        train_data.append({'filename': k, 'label': int(label_val)})

# Extract the clean integer labels for stratification
labels = [x['label'] for x in train_data]

print(f"Total valid images parsed: {len(train_data)}")

# Stratified split to maintain balance across all 12 classes
train_list, val_list = train_test_split(train_data, test_size=0.15, random_state=42, stratify=labels)

train_loader = DataLoader(ParasiteDataset(train_list, TRAIN_DIR, transform_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ParasiteDataset(val_list, TRAIN_DIR, transform_test), batch_size=BATCH_SIZE)
# --- Training Loop ---
model = DinoV2Classifier(NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct = 0, 0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()

    scheduler.step()
    print(f"Loss: {total_loss/len(train_loader):.4f} | Train Acc: {100.*correct/len(train_list):.2f}%")

Detected COCO JSON format. Parsing...
Total valid images parsed: 11000


Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitl14_pretrain.pth


  0%|          | 0.00/1.13G [00:00<?, ?B/s]

...

Epoch 1:   8%|▊         | 48/585 [03:02<35:23,  3.95s/it]

Epoch 1:   8%|▊         | 49/585 [03:06<36:17,  4.06s/it]

Epoch 1:   9%|▊         | 50/585 [03:10<34:38,  3.89s/it]

Epoch 1:   9%|▊         | 51/585 [03:14<35:17,  3.97s/it]

Epoch 1:   9%|▉         | 52/585 [03:18<36:23,  4.10s/it]

In [ ]:
import torch
test_preds=[]
with torch.no_grad():
    for filename in tqdm(sample_sub['filename'], desc="Inference"):
        img_path = os.path.join(TEST_DIR, filename)
        img = Image.open(img_path).convert('RGB')
        img_tensor = transform_test(img).unsqueeze(0).to(device)

        output = model(img_tensor)

        # 1. Apply softmax to convert raw logits into confidence probabilities (0 to 1)
        probabilities = torch.softmax(output, dim=1)

        # 2. Get the highest probability (confidence) and its predicted class
        max_prob, pred = probabilities.max(1)

        # 3. Apply the threshold logic
        if max_prob.item() < 0.7:
            test_preds.append(-1)
        else:
            test_preds.append(pred.item())

# Preserve original CSV structure
sample_sub['label'] = test_preds
sample_sub.to_csv("submission_reject.csv", index=False)
print("Submission saved successfully!")